In [1]:
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
!pip install mordred --no-index --find-links=file:///kaggle/input/mordred-1-2-0-py3-none-any/

Processing /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
Looking in links: file:///kaggle/input/mordred-1-2-0-py3-none-any/
Processing /kaggle/input/mordred-1-2-0-py3-none-any/mordred-1.2.0-py3-none-any.whl
Processing /kaggle/input/mordred-1-2-0-py3-none-any/networkx-2.8.8-py3-none-any.whl (from mordred)
  Attempting uninstall: networkx
    Found existing installation: networkx 3.5
    Uninstalling networkx-3.5:
      Successfully uninstalled networkx-3.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
nx-cugraph-cu12 25.2.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import re
import requests
import joblib, numpy

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import scipy.stats as stats

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [3]:
# Load data
train = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train.csv')
test = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

def improved_tokenize_smiles(smiles):
    """
    Improved SMILES tokenization with better pattern matching
    """
    # More comprehensive pattern for SMILES tokens
    pattern = r'(\[[^\[\]]{1,10}\]|Br|Cl|[bcnops]|[BCNOFPSI]|[()=+\-#@:/\\]|\d+|\.)'
    tokens = re.findall(pattern, smiles)
    
    # Handle remaining characters not caught by pattern
    reconstructed = ''.join(tokens)
    remaining = smiles.replace(reconstructed, '')
    if remaining:
        tokens.extend(list(remaining))
    
    return tokens

def create_smiles_features(train_smiles, test_smiles):
    """
    Create comprehensive SMILES-based features
    """
    # Tokenize all SMILES
    train_tokens = [improved_tokenize_smiles(smiles) for smiles in train_smiles]
    test_tokens = [improved_tokenize_smiles(smiles) for smiles in test_smiles]
    
    # Create vocabulary from all tokens
    all_tokens = []
    for tokens in train_tokens + test_tokens:
        all_tokens.extend(tokens)
    
    vocab = sorted(list(set(all_tokens)))
    vocab_dict = {token: idx for idx, token in enumerate(vocab)}
    
    print(f"Vocabulary size: {len(vocab)}")
    print(f"Max sequence length: {max([len(t) for t in train_tokens + test_tokens])}")
    
    # Method 1: Token counts (bag of tokens)
    def get_token_counts(token_list, vocab_dict):
        features = np.zeros(len(vocab_dict))
        for tokens in token_list:
            for token in tokens:
                if token in vocab_dict:
                    features[vocab_dict[token]] += 1
        return features
    
    train_counts = np.array([get_token_counts([tokens], vocab_dict) for tokens in train_tokens])
    test_counts = np.array([get_token_counts([tokens], vocab_dict) for tokens in test_tokens])
    
    # Method 2: Sequence features (padded sequences)
    max_len = max([len(t) for t in train_tokens + test_tokens])
    max_len = min(max_len, 500)  # Cap at 500 to avoid memory issues
    
    def pad_sequence(tokens, max_len, vocab_dict):
        sequence = []
        for i, token in enumerate(tokens[:max_len]):
            if token in vocab_dict:
                sequence.append(vocab_dict[token])
            else:
                sequence.append(0)  # Unknown token
        
        # Pad with zeros
        while len(sequence) < max_len:
            sequence.append(0)
        
        return sequence
    
    train_sequences = np.array([pad_sequence(tokens, max_len, vocab_dict) for tokens in train_tokens])
    test_sequences = np.array([pad_sequence(tokens, max_len, vocab_dict) for tokens in test_tokens])
    
    # Method 3: Chemical descriptors from SMILES
    def get_chemical_descriptors(smiles):
        """
        Extract chemical descriptors from SMILES string
        """
        features = {}
        
        # Basic molecular features
        features['length'] = len(smiles)
        features['num_atoms'] = len(re.findall(r'[A-Z][a-z]?', smiles))
        features['num_bonds'] = smiles.count('=') + smiles.count('#')
        features['num_rings'] = smiles.count('1') + smiles.count('2') + smiles.count('3') + \
                               smiles.count('4') + smiles.count('5') + smiles.count('6')
        features['num_branches'] = smiles.count('(') + smiles.count(')')
        features['num_aromatic'] = smiles.count('[') + smiles.count(']')
        
        # Atom counts
        features['carbon_count'] = smiles.count('C') + smiles.count('c')
        features['nitrogen_count'] = smiles.count('N') + smiles.count('n')
        features['oxygen_count'] = smiles.count('O') + smiles.count('o')
        features['sulfur_count'] = smiles.count('S') + smiles.count('s')
        features['phosphorus_count'] = smiles.count('P') + smiles.count('p')
        features['fluorine_count'] = smiles.count('F')
        features['chlorine_count'] = smiles.count('Cl')
        features['bromine_count'] = smiles.count('Br')
        
        # Functional groups
        features['alcohol_groups'] = smiles.count('OH')
        features['carboxyl_groups'] = smiles.count('COOH')
        features['ester_groups'] = smiles.count('COO')
        features['ether_groups'] = smiles.count('O') - features['alcohol_groups'] - \
                                  features['carboxyl_groups'] - features['ester_groups']
        
        # Structural complexity
        features['complexity_score'] = features['num_rings'] + features['num_branches'] + \
                                     features['num_bonds'] * 0.5
        
        # Ratios
        total_atoms = max(features['num_atoms'], 1)
        features['heteroatom_ratio'] = (features['nitrogen_count'] + features['oxygen_count'] + 
                                      features['sulfur_count'] + features['phosphorus_count']) / total_atoms
        features['carbon_ratio'] = features['carbon_count'] / total_atoms
        
        return features
    
    # Extract chemical descriptors
    train_descriptors = [get_chemical_descriptors(smiles) for smiles in train_smiles]
    test_descriptors = [get_chemical_descriptors(smiles) for smiles in test_smiles]
    
    descriptor_names = list(train_descriptors[0].keys())
    
    train_desc_df = pd.DataFrame(train_descriptors, columns=descriptor_names)
    test_desc_df = pd.DataFrame(test_descriptors, columns=descriptor_names)
    
    return {
        'vocab': vocab_dict,
        'train_counts': train_counts,
        'test_counts': test_counts,
        'train_sequences': train_sequences,
        'test_sequences': test_sequences,
        'train_descriptors': train_desc_df,
        'test_descriptors': test_desc_df
    }

In [4]:
smiles_features = create_smiles_features(train['SMILES'], test['SMILES'])

Vocabulary size: 100
Max sequence length: 610


In [5]:
def get_ngram_features(tokens, top_ngrams, n=2):
    features = np.zeros(len(top_ngrams))
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        if ngram in top_ngrams:
            features[top_ngrams.index(ngram)] += 1
    return features

def create_ngram_features(token_list, n=2, top_k=1000):
    """
    Create n-gram features from tokenized SMILES
    """
    from collections import Counter
    
    # Generate n-grams
    all_ngrams = []
    for tokens in token_list:
        for i in range(len(tokens) - n + 1):
            ngram = tuple(tokens[i:i+n])
            all_ngrams.append(ngram)
    
    # Get most common n-grams
    ngram_counts = Counter(all_ngrams)
    top_ngrams = [ngram for ngram, _ in ngram_counts.most_common(top_k)]
    
    # Create features    
    def get_ngram_features(tokens, top_ngrams):
        features = np.zeros(len(top_ngrams))
        for i in range(len(tokens) - n + 1):
            ngram = tuple(tokens[i:i+n])
            if ngram in top_ngrams:
                features[top_ngrams.index(ngram)] += 1
        return features
    
    return top_ngrams, [get_ngram_features(tokens, top_ngrams) for tokens in token_list]

In [6]:
# Retokenize for n-grams
train_tokens_list = [improved_tokenize_smiles(smiles) for smiles in train['SMILES']]
test_tokens_list = [improved_tokenize_smiles(smiles) for smiles in test['SMILES']]

# Create bigram and trigram features
top_bigrams, train_bigram_features = create_ngram_features(train_tokens_list + test_tokens_list, n=2, top_k=500)
_, test_bigram_features = create_ngram_features(test_tokens_list, n=2, top_k=500)
test_bigram_features = [get_ngram_features(tokens, top_bigrams) for tokens in test_tokens_list]

train_bigram_features = [get_ngram_features(tokens, top_bigrams, 2) for tokens in train_tokens_list]
test_bigram_features = [get_ngram_features(tokens, top_bigrams, 2) for tokens in test_tokens_list]

# Convert to DataFrames
train_bigram_df = pd.DataFrame(train_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])
test_bigram_df = pd.DataFrame(test_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])

# Combine all features
print("Combining features...")

Combining features...


In [7]:
train_base = train.drop(columns=['SMILES'] if 'SMILES' in train.columns else [])
test_base = test.drop(columns=['SMILES'] if 'SMILES' in test.columns else [])

# Add token count features (most informative)
train_counts_df = pd.DataFrame(smiles_features['train_counts'], 
                              columns=[f'token_count_{i}' for i in range(smiles_features['train_counts'].shape[1])])
test_counts_df = pd.DataFrame(smiles_features['test_counts'], 
                             columns=[f'token_count_{i}' for i in range(smiles_features['test_counts'].shape[1])])

# Convert bigram features to DataFrames
train_bigram_df = pd.DataFrame(train_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])
test_bigram_df = pd.DataFrame(test_bigram_features, columns=[f'bigram_{i}' for i in range(len(top_bigrams))])

# Add chemical descriptors
train_final = pd.concat([
    train_base,
    smiles_features['train_descriptors'],
    train_counts_df,
    train_bigram_df
], axis=1)

test_final = pd.concat([
    test_base,
    smiles_features['test_descriptors'], 
    test_counts_df,
    test_bigram_df
], axis=1)

print(f"Final feature count: {train_final.shape[1]}")

Final feature count: 627


In [8]:
train_split, val_split = train_test_split(
    train_final, test_size=0.07, random_state=42
)

train_split_Tg = train_split.drop(columns = ["FFV", "Tc", "Density", "Rg"]).dropna()
train_split_FFV = train_split.drop(columns = ["Tg", "Tc", "Density", "Rg"]).dropna()
train_split_Tc = train_split.drop(columns = ["Tg", "FFV", "Density", "Rg"]).dropna()
train_split_Density = train_split.drop(columns = ["Tg", "FFV", "Tc", "Rg"]).dropna()
train_split_Rg = train_split.drop(columns = ["Tg", "FFV", "Tc", "Density"]).dropna()

val_split_Tg = train_split.drop(columns = ["FFV", "Tc", "Density", "Rg"]).dropna()
val_split_FFV = train_split.drop(columns = ["Tg", "Tc", "Density", "Rg"]).dropna()
val_split_Tc = train_split.drop(columns = ["Tg", "FFV", "Density", "Rg"]).dropna()
val_split_Density = train_split.drop(columns = ["Tg", "FFV", "Tc", "Rg"]).dropna()
val_split_Rg = train_split.drop(columns = ["Tg", "FFV", "Tc", "Density"]).dropna()

In [9]:
# Improved CatBoost parameters
improved_catboost_params = {
    'iterations': 5000,
    'learning_rate': 0.005,  # Lower learning rate with more iterations
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'min_child_samples': 50,
    'colsample_bylevel': 0.8,  # Feature subsampling
    'od_wait': 100,
    'random_state': 42,
    'od_type': 'Iter',
    'bootstrap_type': 'Bayesian',
    'grow_policy': 'Depthwise',
    'logging_level': 'Silent',
    'loss_function': 'MultiRMSE',
    'use_best_model': True
}

# Train model
print("Training CatBoost Tg model...")
catboost_Tg = CatBoostRegressor(**improved_catboost_params)
catboost_Tg.fit(
    Pool(train_split_Tg.drop(columns = ['Tg']), train_split_Tg['Tg']), 
    eval_set=Pool(val_split_Tg.drop(columns = ['Tg']), val_split_Tg['Tg']), 
    verbose=200, 
    early_stopping_rounds=200
)

print("Training CatBoost FFV model...")
catboost_FFV = CatBoostRegressor(**improved_catboost_params)
catboost_FFV.fit(
    Pool(train_split_FFV.drop(columns = ['FFV']), train_split_FFV['FFV']), 
    eval_set=Pool(val_split_FFV.drop(columns = ['FFV']), val_split_FFV['FFV']), 
    verbose=200, 
    early_stopping_rounds=200
)

print("Training CatBoost Tc model...")
catboost_Tc = CatBoostRegressor(**improved_catboost_params)
catboost_Tc.fit(
    Pool(train_split_Tc.drop(columns = ['Tc']), train_split_Tc['Tc']), 
    eval_set=Pool(val_split_Tc.drop(columns = ['Tc']), val_split_Tc['Tc']), 
    verbose=200, 
    early_stopping_rounds=200
)

print("Training CatBoost Density model...")
catboost_Density = CatBoostRegressor(**improved_catboost_params)
catboost_Density.fit(
    Pool(train_split_Density.drop(columns = ['Density']), train_split_Density['Density']), 
    eval_set=Pool(val_split_Density.drop(columns = ['Density']), val_split_Density['Density']), 
    verbose=200, 
    early_stopping_rounds=200
)

print("Training CatBoost Rg model...")
catboost_Rg = CatBoostRegressor(**improved_catboost_params)
catboost_Rg.fit(
    Pool(train_split_Rg.drop(columns = ['Rg']), train_split_Rg['Rg']), 
    eval_set=Pool(val_split_Rg.drop(columns = ['Rg']), val_split_Rg['Rg']), 
    verbose=200, 
    early_stopping_rounds=200
)

Training CatBoost Tg model...
Training CatBoost FFV model...
Training CatBoost Tc model...
Training CatBoost Density model...
Training CatBoost Rg model...


In [10]:
# Make predictions
print("Making predictions...")

test_predictions_Tg = catboost_Tg.predict(test_final)
test_predictions_FFV = catboost_FFV.predict(test_final)
test_predictions_Tc = catboost_Tc.predict(test_final)
test_predictions_Density = catboost_Density.predict(test_final)
test_predictions_Rg = catboost_Rg.predict(test_final)
print(test_predictions_Tg)

# Create submission
submission = pd.DataFrame({
    'id': test_final['id'],
    'Tg': test_predictions_Tg,
    'FFV': test_predictions_FFV, 
    'Tc': test_predictions_Tc,
    'Density': test_predictions_Density,
    'Rg': test_predictions_Rg
})

print("Feature importance (top 20):")
print("Submission file saved as 'improved_submission.csv'")

Making predictions...
[149.40665596 197.15131445  85.89262027]
Feature importance (top 20):
Submission file saved as 'improved_submission.csv'


In [11]:
submission.to_csv("submission.csv", index=False)
submission

,id,Tg,FFV,Tc,Density,Rg
0,1109053969,149.406656,0.374632,0.226679,1.122117,21.834954
1,1422188626,197.151314,0.378548,0.255008,1.091091,21.481057
2,2032016830,85.892620,0.356334,0.288204,1.155621,22.556437
